# Lab: Functional Encryption from LWE
**Cloud Encryption, Day II, Session III**

Work through the exercises in order; the last one is a homework.

* Every exercise has cells marked `# TODO`: write your code there, then run the **check** cell below it.
* Run the next cell first: it loads the helper library (parameters, sampling, reduction mod $q$) and tests it.
  If it says `No module named 'numpy'`, run `python3 -m pip install numpy` in a terminal and restart the notebook.
* **Toy parameters**: nothing here is secure. Vectors are numpy arrays; `s @ A` is the product $\mathbf s^{\mathsf T}\mathbf A$ and `A @ r` is $\mathbf A\mathbf r$.

In [ ]:
# ===== Helper library (run me first) =====
"""Helper library for the Day II lab (Functional Encryption from LWE).
Toy parameters, NOT secure: for understanding only.  Needs numpy.
Runs in plain Jupyter and in JupyterLite (Pyodide kernel; numpy is loaded automatically)."""
import numpy as np

rng = np.random.default_rng(2026)      # fixed seed: everybody gets the same numbers

# ---- parameters (toy!) -------------------------------------------------------
q = 2**25           # modulus
n = 32              # LWE dimension (secret length)
m = 900             # number of samples (columns of A)
sigma = 1.0         # width of the LWE noise

def mod(x):
    """Centred representative of x mod q, in [-q/2, q/2) (works on numbers and arrays)."""
    return ((np.asarray(x, dtype=np.int64) + q // 2) % q) - q // 2

def unif(shape):
    """Uniform integers mod q, e.g. unif(n) is a vector, unif((n, m)) a matrix."""
    return rng.integers(0, q, shape, dtype=np.int64)

def gauss(shape, s=None):
    """Rounded Gaussian noise of width s (default sigma)."""
    s = sigma if s is None else s
    return np.rint(rng.normal(0, s, shape)).astype(np.int64)

def decode_bit(v):
    """0 if v is closer to 0 than to q/2 (mod q), else 1."""
    return int(abs(int(mod(v))) > q // 4)

# ---- inner-product FE parameters ----------------------------------------------
ell = 8             # vector length
P = 16              # entries of x and y in {0, ..., P-1}
K = 4096            # results live mod K;  K > ell*(P-1)**2 = 1800
Delta = q // K      # scaling factor floor(q/K)

# ---- GSW helpers (homework) ----------------------------------------------------
k = 25                                     # = log2(q)
N = (n + 1) * k
G = np.kron(np.eye(n + 1, dtype=np.int64), 2 ** np.arange(k, dtype=np.int64))   # gadget, (n+1) x N

def Ginv(C):
    """Bit decomposition, lsb first: G @ Ginv(C) == C mod q, with Ginv(C) binary."""
    C = np.asarray(C) % q
    bits = (C[:, None, :] >> np.arange(k)[None, :, None]) & 1
    return bits.reshape(-1, C.shape[1])

def gsw_keygen():
    """Public matrix A (with the LWE row appended) and secret t = (s, -1): t @ A is small."""
    A0, s = unif((n, m)), unif(n)
    b = mod(s @ A0 + gauss(m))
    return np.vstack([A0, b]), np.append(s, -1)

def gsw_enc(A, x):
    R = rng.integers(0, 2, (m, N))
    return mod(A @ R + x * G)

def gsw_dec(t, C):
    w = np.zeros((n + 1, 1), dtype=np.int64); w[-1, 0] = -(q // 2)
    return decode_bit(t @ mod(C @ Ginv(w))[:, 0])

def gsw_noise(t, C, x):
    """Largest entry of t @ C - x * t @ G, relative to q/4 (decryption fails beyond 1)."""
    return float(np.abs(mod(t @ C - x * (t @ G))).max() / (q / 4))

def selftest():
    assert mod(q // 2 + 3) == -(q // 2) + 3 and decode_bit(q // 2) == 1 and decode_bit(5) == 0
    C = unif((n + 1, 3)); assert (mod(G @ Ginv(C) - C) == 0).all()
    print(f"Library OK: numpy {np.__version__}; q = 2^25, n = {n}, m = {m}; IPFE: ell = {ell}, K = {K}")


selftest()

## Exercise 1 — LWE and Regev encryption
* Key generation: uniform $\mathbf A\in\mathbb Z_q^{n\times m}$ and $\mathbf s\in\mathbb Z_q^n$; $\mathbf b^{\mathsf T}=\mathbf s^{\mathsf T}\mathbf A+\mathbf e^{\mathsf T}$ with Gaussian $\mathbf e$. Public key $(\mathbf A,\mathbf b)$, secret key $\mathbf s$.
* Encryption of a bit $\mu$: random binary $\mathbf r\in\{0,1\}^m$; $\mathbf c_0=\mathbf A\mathbf r$, $c_1=\mathbf b^{\mathsf T}\mathbf r+\mu\lfloor q/2\rfloor$.
* Decryption: $c_1-\mathbf s^{\mathsf T}\mathbf c_0$, then `decode_bit`.

Use `unif`, `gauss`, `mod`, `decode_bit` from the library and `rng.integers(0, 2, m)` for $\mathbf r$.

In [ ]:
def regev_keygen():
    raise NotImplementedError  # TODO

def regev_enc(pk, mu):
    raise NotImplementedError  # TODO

def regev_dec(s, c):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
pk, s = regev_keygen()
bits = rng.integers(0, 2, 200)
print("200 random bits decrypt correctly:", all(regev_dec(s, regev_enc(pk, int(mu))) == mu for mu in bits))
c0, c1 = regev_enc(pk, 0)
print("noise c1 - <s,c0> for mu = 0:", int(mod(c1 - s @ c0)), " (q/4 =", q // 4, ")")

**Think:** the noise is $\mathbf e^{\mathsf T}\mathbf r$. How large can it be, at most, with these parameters? How far is that from $q/4$?

## Exercise 2 — Dual Regev, and when decryption fails
Dual Regev (the template of the day):
* Key generation: uniform $\mathbf A$, binary $\mathbf z\in\{0,1\}^m$, $\mathbf u=\mathbf A\mathbf z$. Public key $(\mathbf A,\mathbf u)$, secret key $\mathbf z$.
* Encryption of $\mu$: fresh $\mathbf s$; $\mathbf c_0^{\mathsf T}=\mathbf s^{\mathsf T}\mathbf A+\mathbf e^{\mathsf T}$, $c_1=\mathbf s^{\mathsf T}\mathbf u+e'+\mu\lfloor q/2\rfloor$, with Gaussian noise of width `noise`.
* Decryption: $c_1-\mathbf c_0^{\mathsf T}\mathbf z$, then `decode_bit`.

The check cell then increases the noise until decryption starts to fail.

In [ ]:
def dual_keygen():
    raise NotImplementedError  # TODO

def dual_enc(pk, mu, noise=sigma):
    raise NotImplementedError  # TODO

def dual_dec(z, c):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
pk, z = dual_keygen()
print("200 random bits decrypt correctly:", all(dual_dec(z, dual_enc(pk, int(mu))) == mu for mu in rng.integers(0, 2, 200)))
def failure_rate(noise, trials=300):
    return sum(dual_dec(z, dual_enc(pk, int(mu), noise)) != mu for mu in rng.integers(0, 2, trials)) / trials
for noise in [1, 1e4, 1e5, 2e5, 3e5, 5e5]:
    print(f"noise width {noise:>9g}:  failure rate {failure_rate(noise):.2f}")

**Think:** the decryption noise is $e'-\mathbf e^{\mathsf T}\mathbf z$. Roughly how large is it for noise width $\sigma$ (how many ones does $\mathbf z$ have)? Predict where failures start, and compare with the table.

## Exercise 3 — Inner-product functional encryption [ALS16]
Plaintexts $\mathbf x\in\{0,\dots,P-1\}^\ell$, keys for $\mathbf y\in\{0,\dots,P-1\}^\ell$, results mod $K$ (`ell`, `P`, `K`, `Delta` $=\lfloor q/K\rfloor$ are in the library).
* Setup: uniform $\mathbf A$; short matrix $\mathbf Z\in\{-1,0,1\}^{m\times\ell}$ (use `rng.integers(-1, 2, (m, ell))`); $\mathbf U=\mathbf A\mathbf Z$. $mpk=(\mathbf A,\mathbf U)$, $msk=\mathbf Z$.
* Encryption: fresh $\mathbf s$; $\mathbf c_0^{\mathsf T}=\mathbf s^{\mathsf T}\mathbf A+\mathbf e_0^{\mathsf T}$, $\mathbf c_1^{\mathsf T}=\mathbf s^{\mathsf T}\mathbf U+\mathbf e_1^{\mathsf T}+\Delta\cdot\mathbf x^{\mathsf T}$.
* Key for $\mathbf y$: $\mathbf z_{\mathbf y}=\mathbf Z\mathbf y$.
* Decryption: $d=\mathbf c_1^{\mathsf T}\mathbf y-\mathbf c_0^{\mathsf T}\mathbf z_{\mathbf y}$ (mod $q$); output $\mathrm{round}(d/\Delta)\bmod K$.

In [ ]:
def ipfe_setup():
    raise NotImplementedError  # TODO

def ipfe_enc(mpk, x):
    raise NotImplementedError  # TODO

def ipfe_keygen(Z, y):
    raise NotImplementedError  # TODO

def ipfe_dec(c, y, zy):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
mpk, Z = ipfe_setup()
ok = True
for _ in range(20):
    x, y = rng.integers(0, P, ell), rng.integers(0, P, ell)
    ok &= ipfe_dec(ipfe_enc(mpk, x), y, ipfe_keygen(Z, y)) == int(x @ y)
print("20 random inner products decrypt correctly:", ok)
x, y = rng.integers(0, P, ell), rng.integers(0, P, ell)
print("example: x =", x, " y =", y, " <x,y> =", int(x @ y), " decrypted:", ipfe_dec(ipfe_enc(mpk, x), y, ipfe_keygen(Z, y)))

**Think:** why must $K$ exceed $\ell(P-1)^2$? What goes wrong if it does not?

## Exercise 4 — Risk scores on encrypted patient records
The hospital encrypts each record (8 small values, below) and uploads the ciphertexts.
The KGC gives the analyst one key, for the vector of risk weights. Write
* `hospital_encrypt(mpk, records)`: the list of ciphertexts, one per record (same order);
* `analyst_scores(cts, w, sk_w)`: the list of risk scores, one per ciphertext.

In [ ]:
features = ["age band", "blood pressure", "cholesterol", "BMI band", "smoker", "diabetes", "family history", "sedentary"]
records = {"r001 M. Rossi":    [12,  9,  7,  6, 1, 0, 1, 1],
           "r002 L. Bianchi":  [ 5,  4,  3,  5, 0, 0, 0, 0],
           "r003 G. Verdi":    [14, 11, 10,  8, 1, 1, 1, 1],
           "r004 A. Russo":    [ 9,  7,  9, 10, 0, 1, 0, 1],
           "r005 S. Esposito": [ 3,  5,  4,  4, 1, 0, 0, 0]}
w = [3, 4, 2, 2, 9, 7, 5, 3]                 # risk weights (the analyst's function)

def hospital_encrypt(mpk, records):
    raise NotImplementedError  # TODO

def analyst_scores(cts, w, sk_w):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
mpk, Z = ipfe_setup()                        # the KGC
cts = hospital_encrypt(mpk, records)         # the hospital uploads these
sk_w = ipfe_keygen(Z, w)                     # the analyst's key
for name, score in zip(records, analyst_scores(cts, w, sk_w)):
    print(f"{name:18s} risk score {score:4d}   (in the clear: {int(np.dot(records[name], w))})")

**Think:** (a) What does the cloud learn from the ciphertexts? (b) What does the analyst learn about a record, besides its score?

## Exercise 5 — Too many keys
The analyst convinces the KGC to issue keys for $\ell=8$ different risk scores (`keys` below: a list of pairs `(y, sk_y)`).
Write `recover_record(c, keys)` that recovers the whole record $\mathbf x$ from one ciphertext.
*Hint:* decrypting gives $\langle\mathbf x,\mathbf y_j\rangle$ for every $j$: a linear system $Y\mathbf x=\mathbf v$; use `np.linalg.solve(Y, v)` and round.

In [ ]:
Ys = []
while len(Ys) < ell:                         # 8 linearly independent weight vectors
    y = rng.integers(0, P, ell)
    if np.linalg.matrix_rank(np.array(Ys + [y])) == len(Ys) + 1:
        Ys.append(y)
keys = [(y, ipfe_keygen(Z, y)) for y in Ys]

def recover_record(c, keys):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
for name, c in zip(records, cts):
    print(f"{name:18s} recovered {recover_record(c, keys)}   true {np.array(records[name])}")

**Think:** (a) Is this an attack on the scheme? (b) What should the KGC do? (c) What does the analyst learn with only 7 keys?

## Homework — Noise growth in GSW
The library contains `gsw_keygen`, `gsw_enc`, `gsw_dec`, the gadget `G`, `Ginv` (bit decomposition) and `gsw_noise(t, C, x)`
(the noise of $\mathbf C$ relative to $q/4$; decryption fails beyond 1).
1. Write `gsw_mult(C1, C2)` $=\mathbf C_1\cdot\mathbf G^{-1}(\mathbf C_2)$ (mod $q$).
2. Starting from $\mathbf D=\mathrm{Enc}(1)$, multiply by $\mathrm{Enc}(1)$ again and again; print the depth, `gsw_noise` and the decryption.
3. By which factor does the noise grow per level? Compare with the bound $N+1$ (`N` is in the library).

In [ ]:
def gsw_mult(C1, C2):
    raise NotImplementedError  # TODO

# experiment: your code here